# 04 - Unsupervised Anomaly Detection with Isolation Forest

This notebook implements and evaluates Isolation Forest to detect previously unseen fraud patterns without relying solely on historical fraud labels.

In [1]:
import os, sys, pickle
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath('..'))
from config.config import RAW_DATA_PATH, ISOLATION_FOREST_MODEL_PATH, SCALER_PATH
from src.models.train import evaluate_classifier

df = pd.read_csv(RAW_DATA_PATH).sort_values(by='Time').reset_index(drop=True)
split_idx = int(0.80 * len(df))
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

features = [c for c in df.columns if c not in ['Time', 'Class']]
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

X_train_norm = scaler.transform(train_df[train_df['Class'] == 0][features].values)
X_test = scaler.transform(test_df[features].values)
y_test = test_df['Class'].values

In [2]:
# Train Isolation Forest on normal training transactions
iso = IsolationForest(n_estimators=100, contamination=0.002, random_state=42, n_jobs=-1)
iso.fit(X_train_norm)

iso_scores = -iso.score_samples(X_test)
iso_preds = np.where(iso.predict(X_test) == -1, 1, 0)

metrics = evaluate_classifier('Isolation Forest', y_test, iso_preds, iso_scores)
print('Isolation Forest Anomaly Detection Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

with open(ISOLATION_FOREST_MODEL_PATH, 'wb') as f:
    pickle.dump(iso, f)
print('Saved model to:', ISOLATION_FOREST_MODEL_PATH)

Isolation Forest Anomaly Detection Metrics:
  model: Isolation Forest
  precision: 0.0
  recall: 0.0
  f1_score: 0.0
  pr_auc: 0.0319
  false_positives: 96
  false_negatives: 75
  true_positives: 0
  true_negatives: 56791
  fp_per_1000: 1.69
  total_cost: 38940.0
Saved model to: /Users/yaseen/Data_Analysis/BDP Project/models/isolation_forest.pkl


## Notebook Explanation

### What was done
- Trained an Isolation Forest model exclusively on legitimate historical transactions.
- Evaluated the model's unsupervised anomaly score on the chronological holdout test set.
- Measured detection metrics (Precision, Recall, F1, PR-AUC, FP per 1000 txns, and Business Cost).
- Serialized `models/isolation_forest.pkl`.

### Why it was done
Supervised models can only detect patterns similar to historical training examples. Isolation Forest detects structural anomalies and novel attack vectors without requiring explicit training labels.

### Input
- Normalized legitimate transactions from the training partition of `data/raw/creditcard.csv`.
- Fitted scaler from `models/scaler.pkl`.

### Output
- Serialized model artifact at `models/isolation_forest.pkl`.
- Anomaly detection performance evaluation.

### Connection to next stage
With offline machine learning models established, `05_synthetic_stream_generation.ipynb` generates real-time synthetic transaction streams with customer, merchant, and location semantics.

### Implementation decisions
- Contamination rate was set to 0.002, aligning with the empirical fraud rate (~0.18%).
- The model was trained only on normal transactions to enforce one-class anomaly bounding.